In [ ]:
!pip install scikit-learn

## Librerias

In [ ]:
from sklearn.pipeline import Pipeline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
import numpy as np




## Descripción del Dataset

- Sl_No: (Según google Serial Number) Es un número de serie, sirve para indicar el número de identificiación o la posición consecutiva de un elemento

- Customer Key: Es un id  único para cada cliente

- Avg_Credit_Limit: Es el límite de crédito promedio asignado a un cliente

- Total_Credit_Cards: Es la cantidad total de tarjetas de crédito que un cliente posee

- Total_visits_online: Es la frecuencia con la que el cliente se conecta a la banca en línea

- Total_calls_made: Es la cantidad de llamadas que el cliente ha realiado al soporte del banco

In [ ]:
file_id = '10S8JVFiLfCoRB9mb0NJG5YhITyXbH37B'

download_url = f'https://drive.google.com/uc?export=download&id={file_id}'

data = pd.read_csv(download_url)

In [ ]:
print("\nTipos de datos:")
print(data.dtypes)


Tipos de datos:
Sl_No                  int64
Customer Key           int64
Avg_Credit_Limit       int64
Total_Credit_Cards     int64
Total_visits_bank      int64
Total_visits_online    int64
Total_calls_made       int64
dtype: object


In [ ]:
X = data.drop(columns=["Sl_No", "Customer Key"])

## Extraccion

In [ ]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    """Selecciona/descarta columnas del dataframe crudo."""

    def __init__(self, columns_to_drop=None):
        self.columns_to_drop = columns_to_drop or []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        cols = [c for c in self.columns_to_drop if c in X.columns]
        return X.drop(columns=cols)

## Filtrado

In [ ]:
class RowFilter(BaseEstimator, TransformerMixin):
    """Elimina duplicados, filas con exceso de nulos y valores fuera de rango."""

    def __init__(self, max_null_ratio=0.5, numeric_bounds=None):
        self.max_null_ratio = max_null_ratio
        self.numeric_bounds = numeric_bounds or {}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = X.drop_duplicates()

        null_ratio = X.isnull().mean(axis=1)
        X = X[null_ratio <= self.max_null_ratio]

        for col, (low, high) in self.numeric_bounds.items():
            if col in X.columns:
                X = X[((X[col] >= low) & (X[col] <= high)) | X[col].isnull()]

        return X

## Limpieza

In [ ]:
columnas_a_descartar = ["Sl_No", "Customer Key"]
rangos_numericos = {}

cleaning_pipeline = Pipeline(steps=[
    ('extraccion', FeatureExtractor(columns_to_drop=columnas_a_descartar)),
    ('filtrado', RowFilter(max_null_ratio=0.5, numeric_bounds=rangos_numericos)),
])

data_clean = cleaning_pipeline.fit_transform(data)
print('Shape original:', data.shape, '-> Shape limpio:', data_clean.shape)

Shape original: (660, 7) -> Shape limpio: (649, 5)


## Split de Datos

In [ ]:
target_column = 'Avg_Credit_Limit'
X = data_clean.drop(columns=[target_column])
y = data_clean[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('X_train:', X_train.shape, ' X_test:', X_test.shape)

X_train: (519, 4)  X_test: (130, 4)


## Transformacion

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, make_column_selector(dtype_include=np.number)),
    ('cat', categorical_pipeline, make_column_selector(dtype_include=object)),
])

## Pipeline Final

In [ ]:
model_pipeline = Pipeline(steps=[
    ('preprocesamiento', preprocessor)
])

X_train_transformed = model_pipeline.fit_transform(X_train)
X_test_transformed = model_pipeline.transform(X_test)

print('X_train_transformed:', X_train_transformed.shape)
print('X_test_transformed:', X_test_transformed.shape)

X_train_transformed: (519, 4)
X_test_transformed: (130, 4)


## Diagrama

In [ ]:
full_pipeline = Pipeline(steps=[
    ('extraccion', FeatureExtractor(columns_to_drop=columnas_a_descartar)),
    ('filtrado', RowFilter(max_null_ratio=0.5, numeric_bounds=rangos_numericos)),
    ('preprocesamiento', preprocessor),
])

full_pipeline

Pipeline(steps=[('extraccion',
                 FeatureExtractor(columns_to_drop=['Sl_No', 'Customer Key'])),
                ('filtrado', RowFilter(numeric_bounds={})),
                ('preprocesamiento',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x79b6a5c5d6a0>),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x79b6a76cd850>)]))])